In [1]:
'''
功能:rdkit验证手性立体化学
ligand1:https://www.ebi.ac.uk/chebi/CHEBI:62456
老师给的(手性有问题): C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/Cucurbitadienol#section=3D-Conformer

ligand2(P450环氧):https://www.ebi.ac.uk/chebi/CHEBI:229949
老师给的(手性有问题): C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C
3D:结构：https://pubchem.ncbi.nlm.nih.gov/compound/171037431#section=3D-Conformer

ligand1 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2 = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
'''
from rdkit import Chem
from rdkit.Chem import AllChem

ligand1_teacher = "C/C(C)=C/CC[C@@H](C)C1CC[C@@]2(C)C3CC=C4C(C)(C)[C@@H](O)CCC4[C@]3(C)CC[C@@]21C"
ligand1_chebi = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CCC=C(C)C"
ligand2_teacher = "C[C@H](CCC1C(C)(C)O1)C2CC[C@@]3(C)C4CC=C5C(C)(C)[C@@H](O)CCC5[C@]4(C)CC[C@@]32C"
ligand2_chebi   = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
ligand3_chebi = "C[C@H](CC[C@H](C(C)(C)O)O)[C@H]1CC[C@@]2([C@@]1(CC[C@@]3([C@H]2CC=C4[C@H]3CC[C@@H](C4(C)C)O)C)C)C"

for name, smi in [("teacher", ligand2_teacher), ("chebi", ligand3_chebi)]:
    mol = Chem.MolFromSmiles(smi)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    print(name)
    print("分子式:", Chem.rdMolDescriptors.CalcMolFormula(mol))
    print("规范SMILES:", Chem.MolToSmiles(mol))
    print("手性中心 (原子序号, R/S 或 ?未指定):", centers)
    print()

teacher
分子式: C30H50O2
规范SMILES: C[C@H](CCC1OC1(C)C)C1CC[C@@]2(C)C3CC=C4C(CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, '?'), (9, '?'), (12, 'S'), (14, '?'), (21, 'S'), (25, '?'), (26, 'R'), (30, 'R')]

chebi
分子式: C30H52O3
规范SMILES: C[C@H](CC[C@@H](O)C(C)(C)O)[C@H]1CC[C@@]2(C)[C@@H]3CC=C4[C@@H](CC[C@H](O)C4(C)C)[C@]3(C)CC[C@]12C
手性中心 (原子序号, R/S 或 ?未指定): [(1, 'R'), (4, 'R'), (10, 'R'), (13, 'S'), (14, 'R'), (17, 'R'), (18, 'R'), (22, 'S'), (25, 'S')]



In [ ]:
'''
功能：SMILES转sdf
'''
from rdkit import Chem
from rdkit.Chem import AllChem
smi_chebi  = "[H][C@@]12CC=C3C(C)(C)[C@@H](O)CC[C@@]3([H])[C@]1(C)CC[C@@]1(C)[C@@]2(C)CC[C@]1([H])[C@H](C)CC[C@@H]1OC1(C)C"
for smiles, name in [(ligand2_chebi, "ligand1")]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"{name}: SMILES解析失败，跳过")
        break

    # 检查手性完整性，未指定的中心先亮红灯
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
    centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
    unassigned = [c for c in centers if c[1] == '?']
    if unassigned:
        print(f"{name}存在未指定手性中心: {unassigned}，构象将由算法任意指定，需核实SMILES")

    mol.SetProp("_Name", name)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
    if len(cids) == 0:
        print(f"{name}: 标准ETKDGv3生成失败，尝试随机坐标初始化")
        params.useRandomCoords = True
        cids = AllChem.EmbedMultipleConfs(mol, numConfs=20, params=params)
        if len(cids) == 0:
            print(f"{name}: 3D构象生成彻底失败，跳过该分子")
            break

    # 力场优化，选能量最低的构象
    energies = []
    for cid in cids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(
                mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid
            )
            ff.Minimize(maxIts=2000)
            energies.append((cid, ff.CalcEnergy()))
        except Exception as e:
            print(f"{name}: 构象{cid}优化报错: {e}")

    if not energies:
        print(f"{name}: 所有构象优化失败，跳过")
        break

    best_cid = min(energies, key=lambda x: x[1])[0]
    print(f"{name}: {len(centers)}个手性中心--->{centers}，最优构象能量 = {min(e for _, e in energies):.2f} kcal/mol")

    writer = Chem.SDWriter(f"{name}_input.sdf")
    writer.write(mol, confId=best_cid)
    writer.close()
    print(f"已生成: {name}_input.sdf\n")

ligand1: 9个手性中心 → [(0, 'R'), (7, 'S'), (11, 'S'), (12, 'R'), (16, 'R'), (18, 'S'), (22, 'R'), (23, 'R'), (27, 'S')]，最优构象能量 = 146.39 kcal/mol
已生成: ligand1_input.sdf

